[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/Multimodal-Deep-Learning/blob/main/00_Setup/00_environment_check/00_environment_check.ipynb)

# Module 00: Environment Check

**Goal:** Verify all dependencies are installed and check your compute resources.

---

## Why Environment Setup Matters

Before training multimodal models, you need to know whether your hardware can **fit the model in memory**. GPU memory is the #1 constraint for deep learning.

### GPU Memory Estimation

$$\text{Memory} \approx \text{Parameters} \times \text{Bytes\_per\_param} \times \text{multiplier}$$

The **multiplier** accounts for everything stored during training:

| Component | Multiplier |
|-----------|------------|
| Model weights | 1× |
| Gradients | +1× |
| Optimizer states (Adam: m, v) | +2× |
| **Total for full fine-tuning** | **~5×** |

**Examples for a 7B parameter model:**

| Precision | Calculation | Memory |
|-----------|-------------|--------|
| FP32 | $7\text{B} \times 4 \text{ bytes} \times 5$ | **140 GB** |
| FP16 | $7\text{B} \times 2 \text{ bytes} \times 5$ | **70 GB** |
| QLoRA (NF4) | $7\text{B} \times 0.5 \text{ bytes} + \text{LoRA overhead}$ | **≈ 4 GB** |

This is why **QLoRA** (Module 04) is transformative — it lets you fine-tune 7B models on a single 16 GB GPU by quantizing weights to 4-bit and only training small LoRA adapter matrices.

In [ ]:
# ============================================================
#  Colab Setup (run this cell first if on Google Colab)
# ============================================================
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = "https://github.com/Gaurav14cs17/Multimodal-Deep-Learning.git"
    REPO_DIR = "/content/Multimodal-Deep-Learning"

    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
        !pip install -q -r {REPO_DIR}/requirements.txt

    os.chdir(f"{REPO_DIR}/00_Setup/00_environment_check")
    os.makedirs(f"{REPO_DIR}/assets", exist_ok=True)
    print(f"Colab ready — working in {os.getcwd()}")
else:
    os.makedirs("../assets", exist_ok=True)

In [ ]:
import sys
print(f"Python: {sys.version}")
print(f"Path:   {sys.executable}")

In [ ]:
import torch
import torchvision
import transformers
import peft
import datasets
import accelerate
import matplotlib
import numpy as np

print("=" * 50)
print(f"{'Library Versions':^50}")
print("=" * 50)
print(f"  PyTorch:       {torch.__version__}")
print(f"  TorchVision:   {torchvision.__version__}")
print(f"  Transformers:  {transformers.__version__}")
print(f"  PEFT:          {peft.__version__}")
print(f"  Datasets:      {datasets.__version__}")
print(f"  Accelerate:    {accelerate.__version__}")
print(f"  Matplotlib:    {matplotlib.__version__}")
print(f"  NumPy:         {np.__version__}")
print("=" * 50)

In [ ]:
print("\n" + "=" * 50)
print(f"{'Compute Resources':^50}")
print("=" * 50)

# GPU Check
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"  GPU:           {torch.cuda.get_device_name(0)}")
    print(f"  GPU Memory:    {gpu.total_memory / 1e9:.1f} GB")
    print(f"  CUDA Version:  {torch.version.cuda}")
    print(f"  cuDNN:         {torch.backends.cudnn.version()}")
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    print("  GPU:           Apple MPS (Metal)")
else:
    print("  GPU:           None (CPU only)")
    print("  TIP:           All notebooks work on CPU with small models!")

import os
cpu_count = os.cpu_count()
print(f"  CPU cores:     {cpu_count}")

try:
    import psutil
    ram = psutil.virtual_memory()
    print(f"  RAM:           {ram.total / 1e9:.1f} GB (available: {ram.available / 1e9:.1f} GB)")
except ImportError:
    print("  RAM:           Install psutil for RAM info")

print("=" * 50)

### GFLOPS — Measuring Compute Throughput

The benchmark below times matrix multiplications and reports **GFLOPS** (Giga Floating-Point Operations Per Second):

$$\text{GFLOPS} = \frac{2 \times N^3}{\text{time (seconds)} \times 10^9}$$

**Where does $2N^3$ come from?** Multiplying two $N \times N$ matrices requires $N^2$ output elements, each computed as a dot product of length $N$ (one multiply + one add per element → $2N$ ops). Total: $N^2 \times 2N = 2N^3$.

| Matrix Size | FLOPs | RTX 3060 (~12 TFLOPS) | T4 (~8 TFLOPS) |
|-------------|-------|----------------------|----------------|
| 256×256 | 33M | ~0.003 ms | ~0.004 ms |
| 1024×1024 | 2.1G | ~0.2 ms | ~0.3 ms |
| 2048×2048 | 17G | ~1.4 ms | ~2.1 ms |

Higher GFLOPS = faster GPU. Compare your results to theoretical peak to gauge utilization (real-world is typically 50–80% of peak).

In [ ]:
# Quick GPU benchmark (if available) — with visual chart
import time
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
sizes = [256, 512, 1024, 2048]
times_ms = []
gflops_list = []

print(f"\nMatrix multiplication benchmark on {device}:")
print("-" * 40)
for size in sizes:
    a = torch.randn(size, size, device=device)
    b = torch.randn(size, size, device=device)
    
    if device.type == 'cuda':
        torch.cuda.synchronize()
    
    start = time.time()
    for _ in range(10):
        c = a @ b
    if device.type == 'cuda':
        torch.cuda.synchronize()
    elapsed = (time.time() - start) / 10
    
    gflops = 2 * size**3 / elapsed / 1e9
    times_ms.append(elapsed * 1000)
    gflops_list.append(gflops)
    print(f"  {size}×{size}: {elapsed*1000:.1f} ms ({gflops:.1f} GFLOPS)")

# Plot benchmark results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Compute Benchmark ({device})', fontsize=16, fontweight='bold')

ax = axes[0]
ax.bar([f'{s}×{s}' for s in sizes], times_ms, color='#3498DB', alpha=0.8)
ax.set_ylabel('Time (ms)')
ax.set_title('Matrix Multiply Latency')
for i, v in enumerate(times_ms):
    ax.text(i, v + max(times_ms)*0.02, f'{v:.1f}ms', ha='center', fontsize=9)

ax = axes[1]
ax.bar([f'{s}×{s}' for s in sizes], gflops_list, color='#2ECC71', alpha=0.8)
ax.set_ylabel('GFLOPS')
ax.set_title('Throughput')
for i, v in enumerate(gflops_list):
    ax.text(i, v + max(gflops_list)*0.02, f'{v:.1f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('../assets/benchmark.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Setup complete! Proceed to Module 01.")

### Minimum Requirements

| Resource | Minimum | Recommended | Notes |
|----------|---------|-------------|-------|
| **GPU** | None (CPU works) | Colab T4 (16 GB) or RTX 3060 (12 GB) | All notebooks run on CPU with small models |
| **RAM** | 8 GB | 16 GB | Large batches in Module 03 need more |
| **Disk** | 5 GB free | 10 GB | For downloaded pretrained models (ViT, BERT, CLIP) |

> **Colab free tier:** T4 GPU with 16 GB VRAM is sufficient for all notebooks through Module 05 (with QLoRA for large models). CPU-only mode works everywhere but training cells will be slower.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(16, 10))
ax.set_xlim(0, 16)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('Your Learning Roadmap', fontsize=22, fontweight='bold', pad=20)

def box(ax, x, y, w, h, label, color, fs=9):
    b = FancyBboxPatch((x-w/2, y-h/2), w, h, boxstyle="round,pad=0.15",
                       facecolor=color, edgecolor='#2C3E50', lw=2, alpha=0.9)
    ax.add_patch(b)
    ax.text(x, y, label, ha='center', va='center', fontsize=fs, fontweight='bold', color='white')

def arrow(ax, s, e):
    ax.annotate('', xy=e, xytext=s, arrowprops=dict(arrowstyle='->', color='#2C3E50', lw=2))

modules = [
    (3, 8.5, 'Module 00\nSetup', '#95A5A6', '15 min'),
    (3, 7, 'Module 01\nMultimodal\nFoundations', '#E74C3C', '1-2 hrs'),
    (8, 7, 'Module 02\nVision-Language\nModels (CLIP)', '#3498DB', '2-3 hrs'),
    (13, 7, 'Module 03\nTraining\nStrategies', '#F39C12', '2-3 hrs'),
    (5.5, 4, 'Module 04\nFinetuning\nLoRA / QLoRA', '#E74C3C', '3-4 hrs'),
    (10.5, 4, 'Module 05\nAdvanced\nLLaVA + Deploy', '#2ECC71', '2-3 hrs'),
]

for x, y, label, color, time_str in modules:
    box(ax, x, y, 3.8, 1.5, label, color, fs=10)
    ax.text(x, y - 1.0, time_str, ha='center', fontsize=9, color='gray', style='italic')

arrow(ax, (3, 7.9), (3, 7.8))
arrow(ax, (4.9, 7), (6.1, 7))
arrow(ax, (9.9, 7), (11.1, 7))
arrow(ax, (5, 6.2), (5.5, 4.9))
arrow(ax, (13, 6.2), (10.5, 4.9))
arrow(ax, (7.4, 4), (8.6, 4))

# Key focus areas
ax.text(8, 2, 'YOUR FOCUS', fontsize=16, ha='center', fontweight='bold', color='#E74C3C')
focus = ['Training from scratch (contrastive, multi-objective)',
         'LoRA / QLoRA finetuning (low compute)',
         'Visual diagrams at every step']
for i, txt in enumerate(focus):
    ax.text(8, 1.4 - i*0.5, f'→ {txt}', fontsize=11, ha='center', color='#2C3E50')

plt.tight_layout()
plt.savefig('../assets/roadmap.png', dpi=150, bbox_inches='tight')
plt.show()